In [ ]:
import lzma
import pickle
import re
import unicodedata

import pandas as pd

In [3]:
with lzma.open("../../data/cleaned/starling_cleaned.pkl.xz", "rb") as f:
    starling = pickle.load(f)

with lzma.open("../../data/cleaned/unihan_cleaned.pkl.xz", "rb") as f:
    unihan = pickle.load(f)

In [ ]:
def take_non_missing_value(row):
    return "; ".join({unicodedata.normalize("NFC", str(value)) for value in row if pd.notna(value)})

merged = unihan.merge(starling, on="Character", how="outer", suffixes=("_unihan", "_starling"))

merged["definition"]      = merged[["kDefinition", "English meaning"]].apply(take_non_missing_value, axis=1)
merged["pinyin"]          = merged[["kMandarin", "Modern (Beijing) reading"]].apply(take_non_missing_value, axis=1)
merged["decomposed_pinyin"] = merged[["decomposed_pinyin_unihan", "decomposed_pinyin_starling"]].apply(take_non_missing_value, axis=1)
merged["toneless_pinyin"] = merged.decomposed_pinyin.apply(lambda x: "".join(re.findall(r'[a-z]+', x)))

merged = merged.rename(columns={
    "Character":   "hanzi",
    "kHangul":     "hangul",
    "kKorean":     "anglo_hangul",
    "kJapanese":   "katakana",
    "kJapaneseOn": "anglo_katakana",
})[["hanzi", "definition", "pinyin", "decomposed_pinyin", "toneless_pinyin", "hangul", "anglo_hangul", "katakana", "anglo_katakana"]]

merged